# Music Annotation Project V8.7
## Notebook 50: Asymmetric Observable Residual-Mask M5 Challenger

**Runtime:** GPU preferred for the local-spatial challenger; CPU permitted for the scalar baseline and audit stages.

**Status:** development-only experiment. This notebook does not authorise production use.

### Revised scientific question
Can observable-only local spatial evidence, evaluated out of fold by source group, safely identify coherent **REMOVE** and **PROTECT** subregions within frozen **M3**, so that reconstruction burden is reduced without structural harm, while every unsupported, invalid or uncertain location retains exact M3 treatment?

### Fixed interpretation after Notebook 49 / 49R4
- `REMOVE` is the primary challenger.
- `PROTECT` is a stricter secondary challenger.
- There is no required `RECONSTRUCT` reference class inside the completed M3 decomposition.
- No assertion means exact M3 fallback.
- M4 is diagnostic only and is never an inference input or direct target.
- Whole M3 components are not classification units. Decisions are made from local evidence and asserted only on coherent page-global 8-connected candidate subregions.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow scikit-learn scipy matplotlib joblib torch torchvision opencv-python-headless


## 1. Configuration and frozen pre-registration

Numerical values below are explicit Notebook 50 commissioning choices. They are not inferred from the attractive 49R4 univariate AUCs. Any change after outcomes are inspected creates a new notebook version.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, os, random, shutil, sys, platform, gc, time
import numpy as np
import pandas as pd
import cv2, joblib
from scipy import ndimage
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
NB49_ROOT = ROOT/'observed_print_classifier_v8_7_notebook49'
NB49R4_ROOT = ROOT/'observed_print_classifier_v8_7_notebook49r4'
OUTPUT_ROOT = ROOT/'observable_residual_mask_m5_v8_7'

NB49_RUN_OVERRIDE = None
NB49R4_RUN_OVERRIDE = None
SEED = 42
BOOTSTRAP_RESAMPLES = 10000
CONNECTIVITY = 8
MIN_REMOVE_RUN = 2
MIN_PROTECT_RUN = 2

# Separate operating rules. These are development pre-registration choices.
REMOVE_PROB_THRESHOLD = 0.995
PROTECT_PROB_THRESHOLD = 0.999
REMOVE_REGION_MIN_MEAN = 0.995
PROTECT_REGION_MIN_MEAN = 0.999
REMOVE_REGION_MIN_PIXEL_FRACTION = 1.00
PROTECT_REGION_MIN_PIXEL_FRACTION = 1.00

# Catastrophe definitions are intentionally asymmetric.
FALSE_REMOVE_CATASTROPHE_PIXELS = 20
FALSE_PROTECT_CATASTROPHE_PIXELS = 20
FALSE_REMOVE_COMPONENT_FRACTION = 0.10
FALSE_PROTECT_COMPONENT_FRACTION = 0.10

# Development pass/fail proposals. Freeze before execution.
MAX_FALSE_REMOVE_RATE = 0.005
MAX_FALSE_REMOVE_WILSON95 = 0.010
MAX_FALSE_PROTECT_RATE = 0.005
MAX_FALSE_PROTECT_WILSON95 = 0.010
MIN_REMOVE_SELECTED_PIXELS = 500
MIN_REMOVE_BENEFITING_PAGES = 10
MIN_REMOVE_BENEFITING_GROUPS = 3
MIN_PROTECT_SELECTED_PIXELS = 50
MIN_PROTECT_BENEFITING_PAGES = 5
MIN_PROTECT_BENEFITING_GROUPS = 2

FORBIDDEN_PATTERNS = ('clean','truth','outcome','failure','reconstruct','m4','target','label','class')
IDENTIFIERS = {'page','source_group','y','x','pixel_id','region_id','parent_component_id','fold'}

RUN = datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook50_v8_7_asymmetric_m5'
OUT = OUTPUT_ROOT/RUN
INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MODELS=OUT/'models'; CALIBRATORS=OUT/'calibrators'
AUDIT=OUT/'audit'; MAPS=OUT/'maps'; VISUAL=OUT/'visual_audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MODELS,CALIBRATORS,AUDIT,MAPS,VISUAL,UPLOAD]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Source resolution, integrity and immutable evidence reuse

The notebook resolves one completed Notebook 49 run and one compatible 49R4 run. It does not rebuild Notebook 45 masks or truth labels. Required source tables are copied into the run before modelling.


In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()


# ============================================================
# EXACT DURABLE CONTRACTS FROM NOTEBOOKS 49R3 AND 49R4
# ============================================================

NB49_REQUIRED = [
    '02_observable_lock/candidate_lock.json',
    '02_observable_lock/feature_schema.json',
    '03_truth_unlock/pixel_oracle_gap.parquet',
    '03_truth_unlock/region_oracle_gap.parquet',
    'notebook49_verdict.json',
    'experiment_identity.json',
]

NB49R4_REQUIRED = [
    'notebook49r4_verdict.json',
    'experiment_identity.json',
    'source_record.json',

    '02_tables/truth_connected_subregions.parquet',
    '02_tables/truth_connected_subregion_membership.parquet',
    '02_tables/subregion_observable_features.parquet',
    '02_tables/subregion_observable_feature_allowlist.csv',
    '02_tables/subregion_coherence_summary.csv',
    '02_tables/subregion_univariate_separability.csv',

    '03_m4_semantics/m4_semantics_summary.json',
]


def complete_runs(root, required):
    root = Path(root)

    if not root.exists():
        return []

    hits = []

    # First inspect the normal layout:
    # <root>/<timestamped_run>/
    for p in root.iterdir():
        if p.is_dir() and all((p / rel).exists() for rel in required):
            hits.append(p)

    # Fail-safe recovery if a completed run has subsequently been
    # placed beneath another directory.
    if not hits:
        for p in root.rglob('*'):
            if p.is_dir() and all((p / rel).exists() for rel in required):
                hits.append(p)

    # De-duplicate while preserving Path objects.
    unique = {}
    for p in hits:
        unique[str(p.resolve())] = p

    return sorted(
        unique.values(),
        key=lambda p: p.stat().st_mtime
    )


def resolve_run(override, root, required, label):
    if override is not None:
        run = Path(override)

        missing = [
            rel for rel in required
            if not (run / rel).exists()
        ]

        if missing:
            raise RuntimeError(
                f'{label} override is incomplete.\n'
                f'Run: {run}\n'
                f'Missing: {missing}'
            )

        return run

    hits = complete_runs(root, required)

    if not hits:
        children = (
            [str(p) for p in sorted(Path(root).iterdir())
             if p.is_dir()]
            if Path(root).exists()
            else []
        )

        raise RuntimeError(
            f'No complete {label} run found beneath:\n'
            f'  {root}\n\n'
            f'Expected relative files:\n'
            + '\n'.join(f'  - {x}' for x in required)
            + '\n\n'
            f'Direct child directories seen:\n'
            + '\n'.join(f'  - {x}' for x in children[:30])
            + '\n\n'
            f'If the run was moved, set the corresponding '
            f'*_RUN_OVERRIDE to the exact completed run directory.'
        )

    return hits[-1]


# ============================================================
# RESOLVE 49R4 FIRST
# ============================================================

NB49R4 = resolve_run(
    NB49R4_RUN_OVERRIDE,
    NB49R4_ROOT,
    NB49R4_REQUIRED,
    'Notebook 49R4'
)

print('Notebook 49R4:', NB49R4)


# ============================================================
# RESOLVE NOTEBOOK 49 THROUGH 49R4 LINEAGE
#
# Do NOT independently choose "latest Notebook 49" unless the
# lineage stored by 49R4 cannot be resolved.
# ============================================================

r4_source_record = json.loads(
    (NB49R4 / 'source_record.json').read_text()
)

r4_identity = json.loads(
    (NB49R4 / 'experiment_identity.json').read_text()
)

recorded_nb49 = (
    r4_source_record.get('source_notebook49')
    or r4_identity.get('source_notebook49')
)

if NB49_RUN_OVERRIDE is not None:
    NB49 = Path(NB49_RUN_OVERRIDE)

elif recorded_nb49:
    NB49 = Path(recorded_nb49)

else:
    # Defensive fallback only.
    hits = complete_runs(
        NB49_ROOT,
        NB49_REQUIRED
    )

    NB49 = hits[-1] if hits else None


if NB49 is None:
    raise RuntimeError(
        'Notebook 49R4 does not identify its Notebook 49 source '
        'and no complete Notebook 49 run could be resolved.'
    )


missing49 = [
    rel for rel in NB49_REQUIRED
    if not (NB49 / rel).exists()
]

if missing49:
    raise RuntimeError(
        'The Notebook 49 run recorded by Notebook 49R4 '
        'is incomplete or has moved.\n'
        f'Recorded path: {NB49}\n'
        f'Missing: {missing49}\n\n'
        'If the run was moved without changing its contents, '
        'set NB49_RUN_OVERRIDE to its present location.'
    )


# ============================================================
# VERIFY 49R4 VERDICT
# ============================================================

verdict49r4 = json.loads(
    (NB49R4 / 'notebook49r4_verdict.json').read_text()
)

allowed_r4_verdicts = {
    'REMOVE_AND_PROTECT_SUPPORT_SEPARATE_NOTEBOOK50_PREREGISTRATION',
    'ONE_TARGET_SUPPORTS_NARROW_NOTEBOOK50_PREREGISTRATION',
}

if verdict49r4.get('verdict') not in allowed_r4_verdicts:
    raise RuntimeError(
        'Notebook 49R4 does not authorise Notebook 50 fitting.\n'
        f'Verdict: {verdict49r4.get("verdict")}'
    )


# ============================================================
# VERIFY NOTEBOOK 49 IDENTITY
# ============================================================

lock49 = json.loads(
    (NB49 / '02_observable_lock/candidate_lock.json').read_text()
)

identity49 = json.loads(
    (NB49 / 'experiment_identity.json').read_text()
)

verdict49 = json.loads(
    (NB49 / 'notebook49_verdict.json').read_text()
)


recorded_lock = (
    r4_source_record.get('candidate_lock_id')
    or r4_identity.get('source_notebook49_lock_id')
)

if recorded_lock is not None:
    if lock49.get('lock_id') != recorded_lock:
        raise RuntimeError(
            'Notebook 49 lineage mismatch.\n'
            f'49R4 records lock: {recorded_lock}\n'
            f'Resolved 49 lock: {lock49.get("lock_id")}'
        )


print('Notebook 49:', NB49)
print('Notebook 49 lock:', lock49.get('lock_id'))
print('Notebook 49R4 verdict:', verdict49r4.get('verdict'))


## 3. Load locked M3 pixel population and enforce the predictor firewall

Expected Notebook 49 pixel columns include page/source identity, coordinates, truth-side class and observable features. The notebook rejects missing identities, duplicate coordinates, non-finite predictor arrays and any predictor whose name indicates truth or clean-derived lineage.


In [ ]:
pixel_path=NB49/'03_truth_unlock/pixel_oracle_gap.parquet'
df=pd.read_parquet(pixel_path)
required={'page','source_group','y','x'}
missing=required-set(df.columns)
if missing: raise RuntimeError(f'Pixel table missing identities: {sorted(missing)}')
label_col=next((c for c in ['truth_class','diagnostic_class','class'] if c in df.columns),None)
if label_col is None: raise RuntimeError('No truth-side diagnostic class column found.')
df[label_col]=df[label_col].astype(str).str.upper()
if df.duplicated(['page','y','x']).any(): raise RuntimeError('Duplicate page/y/x rows.')
if not set(df[label_col].unique()).issubset({'REMOVE','PROTECT','UNCERTAIN','RECONSTRUCT'}):
    raise RuntimeError(f'Unexpected classes: {sorted(df[label_col].unique())}')

numeric=list(df.select_dtypes(include=[np.number,bool]).columns)
predictors=[]
for c in numeric:
    if c in IDENTIFIERS: continue
    if any(p in c.lower() for p in FORBIDDEN_PATTERNS): continue
    q=pd.to_numeric(df[c],errors='coerce')
    if np.isfinite(q).mean()<0.95 or q.dropna().nunique()<2: continue
    predictors.append(c)
if not predictors: raise RuntimeError('No eligible observable predictors remain.')

firewall=pd.DataFrame({'predictor':predictors})
firewall.to_csv(INPUTS/'predictor_allowlist.csv',index=False)
(INPUTS/'predictor_schema.json').write_text(json.dumps({'ordered_predictors':predictors,'forbidden_patterns':FORBIDDEN_PATTERNS},indent=2))
print('Rows:',len(df),'pages:',df.page.nunique(),'groups:',df.source_group.nunique(),'predictors:',len(predictors))
print(df[label_col].value_counts())


## 4. Source-group OOF folds

Each source group is held out intact. REMOVE and PROTECT use exactly the same outer folds. Page identity and source-group identity are evaluation/grouping variables, not predictors.


In [ ]:
groups=df.source_group.astype(str)
if groups.nunique()<3: raise RuntimeError('At least three source groups are required.')
splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique()))
fold=np.full(len(df),-1,int); split_rows=[]
for f,(tr,te) in enumerate(splitter.split(df,groups=groups),1):
    train_groups=set(groups.iloc[tr]); test_groups=set(groups.iloc[te]); overlap=train_groups&test_groups
    if overlap: raise RuntimeError(f'Fold {f} source leakage: {sorted(overlap)}')
    fold[te]=f
    split_rows.append({'fold':f,'train_rows':len(tr),'test_rows':len(te),'train_groups':len(train_groups),'test_groups':len(test_groups),'test_group_names':'|'.join(sorted(test_groups))})
if (fold<1).any(): raise RuntimeError('Incomplete fold assignment.')
df['fold']=fold
pd.DataFrame(split_rows).to_csv(INPUTS/'source_group_split_audit.csv',index=False)
df[['page','source_group','y','x','fold']].to_parquet(INPUTS/'candidate_fold_manifest.parquet',index=False)


## 5. Paired asymmetric OOF models

For each target, the scalar baseline and bounded non-linear challenger are fitted only on training folds. The primary executable baseline is regularised logistic regression. This notebook intentionally does not convert the 49R4 feature contrasts into a handcrafted routing rule.


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def make_models():
    return {
      'scalar_logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=4000,random_state=SEED))]),
      'local_nonlinear':Pipeline([('impute',SimpleImputer(strategy='median')),('model',HistGradientBoostingClassifier(max_iter=300,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=2.0,random_state=SEED))])
    }

X=df[predictors].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
results=[]; fold_results=[]
for target in ['REMOVE','PROTECT']:
    y=(df[label_col]==target).astype(int)
    for model_name,model in make_models().items():
        raw=np.full(len(df),np.nan)
        for f in sorted(df.fold.unique()):
            tr=np.where(df.fold.to_numpy()!=f)[0]; te=np.where(df.fold.to_numpy()==f)[0]
            if y.iloc[tr].nunique()<2: continue
            model.fit(X.iloc[tr],y.iloc[tr]); raw[te]=model.predict_proba(X.iloc[te])[:,1]
            yt=y.iloc[te]; pt=raw[te]
            fold_results.append({'target':target,'model':model_name,'fold':int(f),'rows':len(te),'positives':int(yt.sum()),'roc_auc':roc_auc_score(yt,pt) if yt.nunique()==2 else np.nan,'average_precision':average_precision_score(yt,pt) if yt.sum() else np.nan,'brier':brier_score_loss(yt,pt)})
        valid=np.isfinite(raw)
        results.append({'target':target,'model':model_name,'rows':int(valid.sum()),'positives':int(y[valid].sum()),'roc_auc':roc_auc_score(y[valid],raw[valid]) if y[valid].nunique()==2 else np.nan,'average_precision':average_precision_score(y[valid],raw[valid]) if y[valid].sum() else np.nan,'brier':brier_score_loss(y[valid],raw[valid])})
        df[f'raw_{target.lower()}_{model_name}']=raw
pd.DataFrame(results).to_csv(TABLES/'model_summary.csv',index=False)
pd.DataFrame(fold_results).to_csv(TABLES/'fold_model_results.csv',index=False)
display(pd.DataFrame(results))


## 6. Cross-fitted calibration and model selection

Calibration is fitted from OOF ranking scores outside each held-out fold. The selected arm for each target is the arm with the best average precision subject to finite complete OOF predictions. Model selection is target-specific.


In [ ]:
model_summary=pd.read_csv(TABLES/'model_summary.csv')
selected_models={}
for target in ['REMOVE','PROTECT']:
    q=model_summary[(model_summary.target==target)&model_summary.average_precision.notna()].sort_values(['average_precision','roc_auc'],ascending=False)
    if q.empty: raise RuntimeError(f'No valid {target} model.')
    selected_models[target]=q.iloc[0].model

cal_inventory=[]
for target,model_name in selected_models.items():
    raw=df[f'raw_{target.lower()}_{model_name}'].to_numpy(float)
    y=(df[label_col]==target).astype(int)
    calibrated=np.full(len(df),np.nan)
    for f in sorted(df.fold.unique()):
        tr=np.where(df.fold.to_numpy()!=f)[0]; te=np.where(df.fold.to_numpy()==f)[0]
        keep=np.isfinite(raw[tr])
        if keep.sum()<20 or y.iloc[tr][keep].nunique()<2: continue
        iso=IsotonicRegression(out_of_bounds='clip').fit(raw[tr][keep],y.iloc[tr].to_numpy()[keep])
        calibrated[te]=iso.predict(raw[te])
        path=CALIBRATORS/f'{target.lower()}_fold{int(f)}_isotonic.joblib'; joblib.dump(iso,path)
        cal_inventory.append({'target':target,'fold':int(f),'path':str(path),'sha256':sha256_file(path)})
    if not np.isfinite(calibrated).all(): raise RuntimeError(f'Incomplete calibrated {target} probabilities.')
    df[f'p_{target.lower()}']=calibrated
pd.DataFrame(cal_inventory).to_csv(AUDIT/'calibrator_inventory.csv',index=False)
(OUT/'selected_models.json').write_text(json.dumps(selected_models,indent=2))
print(selected_models)


## 7. Page-global connected-subregion assertions

Provisional high-confidence pixels are grouped page-globally with 8-connectivity. A candidate region is asserted only if it passes the target's frozen minimum size, minimum mean probability and all-pixels-above-threshold requirement. Overlap resolves fail-closed: PROTECT and REMOVE conflict becomes M3 fallback.


In [ ]:
def region_assertions(frame,target,threshold,min_mean,min_run):
    asserted=np.zeros(len(frame),bool); region_id=np.full(len(frame),'',object); rows=[]
    for page,g in frame.groupby('page',sort=False):
        idx=g.index.to_numpy(); h=int(g.y.max())+1; w=int(g.x.max())+1
        candidate=g[f'p_{target.lower()}'].to_numpy()>=threshold
        m=np.zeros((h,w),np.uint8); m[g.y.to_numpy(int)[candidate],g.x.to_numpy(int)[candidate]]=1
        n,lab,stats,_=cv2.connectedComponentsWithStats(m,CONNECTIVITY)
        coord_to_index={(int(frame.at[i,'y']),int(frame.at[i,'x'])):i for i in idx}
        for cid in range(1,n):
            ys,xs=np.where(lab==cid); ids=[coord_to_index[(int(y),int(x))] for y,x in zip(ys,xs) if (int(y),int(x)) in coord_to_index]
            if not ids: continue
            probs=frame.loc[ids,f'p_{target.lower()}'].to_numpy(float); size=len(ids); mean=float(probs.mean())
            passes=size>=min_run and mean>=min_mean and float((probs>=threshold).mean())>=1.0
            rid=f'{page}:{target}:r{cid:05d}'
            if passes: asserted[ids]=True; region_id[ids]=rid
            rows.append({'page':page,'source_group':str(frame.loc[ids[0],'source_group']),'target':target,'region_id':rid,'pixels':size,'mean_probability':mean,'minimum_probability':float(probs.min()),'passes':passes})
    return asserted,region_id,pd.DataFrame(rows)

safe_remove,remove_region,remove_regions=region_assertions(df,'REMOVE',REMOVE_PROB_THRESHOLD,REMOVE_REGION_MIN_MEAN,MIN_REMOVE_RUN)
safe_protect,protect_region,protect_regions=region_assertions(df,'PROTECT',PROTECT_PROB_THRESHOLD,PROTECT_REGION_MIN_MEAN,MIN_PROTECT_RUN)
conflict=safe_remove&safe_protect
safe_remove[conflict]=False; safe_protect[conflict]=False

df['safe_remove']=safe_remove; df['safe_protect']=safe_protect; df['assertion_conflict']=conflict
df['remove_region_id']=remove_region; df['protect_region_id']=protect_region
df['m5_reason']=np.where(safe_remove,'SAFE_REMOVE',np.where(safe_protect,'SAFE_PROTECT',np.where(conflict,'CONFLICT_FALLBACK_M3','FALLBACK_M3')))
remove_regions.to_csv(TABLES/'remove_candidate_regions.csv',index=False)
protect_regions.to_csv(TABLES/'protect_candidate_regions.csv',index=False)
df.to_parquet(TABLES/'all_m3_oof_predictions_and_decisions.parquet',index=False)
print(df.m5_reason.value_counts())


## 8. Asymmetric truth-side safety, benefit and catastrophe accounting

- False REMOVE: selected REMOVE pixels whose diagnostic class is PROTECT.
- False PROTECT: selected PROTECT pixels whose diagnostic class is REMOVE.
- Truth UNCERTAIN selections are reported separately and do not count as demonstrated benefit.
- Unselected pixels are intended fallback and remain M3.


In [ ]:
def wilson_upper(errors,n,z=1.96):
    if n==0:return np.nan
    p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d

def target_summary(target,selection):
    truth=df[label_col].to_numpy(); selected=np.asarray(selection,bool)
    correct=int(np.sum(selected&(truth==target)))
    opposite='PROTECT' if target=='REMOVE' else 'REMOVE'
    false=int(np.sum(selected&(truth==opposite)))
    uncertain=int(np.sum(selected&(truth=='UNCERTAIN')))
    denom=correct+false
    return {'target':target,'selected':int(selected.sum()),'correct':correct,'false_opposite':false,'truth_uncertain':uncertain,'false_rate':false/max(denom,1),'wilson95_upper_false_rate':wilson_upper(false,denom),'pages':int(df.loc[selected,'page'].nunique()),'source_groups':int(df.loc[selected,'source_group'].nunique())}

summaries=[target_summary('REMOVE',safe_remove),target_summary('PROTECT',safe_protect)]
summary_df=pd.DataFrame(summaries); summary_df.to_csv(TABLES/'asymmetric_target_summary.csv',index=False)

region_eval=[]
for target,regions,selcol,ridcol in [('REMOVE',remove_regions,'safe_remove','remove_region_id'),('PROTECT',protect_regions,'safe_protect','protect_region_id')]:
    for rid,g in df[df[selcol]].groupby(ridcol):
        counts=g[label_col].value_counts(); correct=int(counts.get(target,0)); opposite='PROTECT' if target=='REMOVE' else 'REMOVE'; false=int(counts.get(opposite,0)); unc=int(counts.get('UNCERTAIN',0))
        frac=false/max(len(g),1); catastrophe=bool(false >= (FALSE_REMOVE_CATASTROPHE_PIXELS if target=='REMOVE' else FALSE_PROTECT_CATASTROPHE_PIXELS) or frac >= (FALSE_REMOVE_COMPONENT_FRACTION if target=='REMOVE' else FALSE_PROTECT_COMPONENT_FRACTION))
        region_eval.append({'target':target,'region_id':rid,'page':g.page.iloc[0],'source_group':g.source_group.iloc[0],'pixels':len(g),'correct':correct,'false_opposite':false,'truth_uncertain':unc,'false_fraction':frac,'catastrophic':catastrophe})
region_df=pd.DataFrame(region_eval,columns=['target','region_id','page','source_group','pixels','correct','false_opposite','truth_uncertain','false_fraction','catastrophic'])
region_df.to_csv(TABLES/'selected_region_truth_evaluation.csv',index=False)
region_df[region_df.catastrophic==True].to_csv(TABLES/'catastrophic_regions.csv',index=False)
display(summary_df)


## 9. Page-clustered uncertainty and pass/fail gates

REMOVE and PROTECT may pass or fail independently. A combined M5 is permitted only from target arms that pass their own integrity, safety, stability and coverage gates. Safety-passing but low-coverage arms are retained as research baselines and are not forced to lower thresholds.


In [ ]:
def bootstrap_false_rate(target,selection):
    page_rows=[]; opposite='PROTECT' if target=='REMOVE' else 'REMOVE'
    for page,g in df.assign(_sel=selection).groupby('page'):
        s=g._sel.to_numpy(bool); t=g[label_col].to_numpy(); page_rows.append({'page':page,'correct':int(np.sum(s&(t==target))),'false':int(np.sum(s&(t==opposite)))})
    q=pd.DataFrame(page_rows).set_index('page'); pages=q.index.to_numpy(); rng=np.random.default_rng(SEED); vals=[]
    for _ in range(BOOTSTRAP_RESAMPLES):
        z=q.loc[rng.choice(pages,len(pages),replace=True)]; vals.append(z.false.sum()/max(z.correct.sum()+z.false.sum(),1))
    return float(np.quantile(vals,.975))

for s in summaries: s['page_bootstrap_false_rate_95_high']=bootstrap_false_rate(s['target'],safe_remove if s['target']=='REMOVE' else safe_protect)
summary_df=pd.DataFrame(summaries)
cat_by_target=region_df.groupby('target').catastrophic.sum().to_dict() if len(region_df) else {}

def gate_target(target):
    s=summary_df[summary_df.target==target].iloc[0]
    groups=df.loc[df['safe_'+target.lower()],'source_group'].nunique(); pages=df.loc[df['safe_'+target.lower()],'page'].nunique()
    if target=='REMOVE': max_rate,max_w,min_sel,min_pages,min_groups=MAX_FALSE_REMOVE_RATE,MAX_FALSE_REMOVE_WILSON95,MIN_REMOVE_SELECTED_PIXELS,MIN_REMOVE_BENEFITING_PAGES,MIN_REMOVE_BENEFITING_GROUPS
    else: max_rate,max_w,min_sel,min_pages,min_groups=MAX_FALSE_PROTECT_RATE,MAX_FALSE_PROTECT_WILSON95,MIN_PROTECT_SELECTED_PIXELS,MIN_PROTECT_BENEFITING_PAGES,MIN_PROTECT_BENEFITING_GROUPS
    return {
      'finite_predictions':bool(np.isfinite(df[f'p_{target.lower()}']).all()),
      'zero_catastrophic_regions':int(cat_by_target.get(target,0))==0,
      'false_rate':float(s.false_rate)<=max_rate,
      'wilson95_upper':float(s.wilson95_upper_false_rate)<=max_w,
      'minimum_selected_pixels':int(s.selected)>=min_sel,
      'minimum_pages':pages>=min_pages,
      'minimum_source_groups':groups>=min_groups,
    }

gates={'REMOVE':gate_target('REMOVE'),'PROTECT':gate_target('PROTECT')}
passes={k:all(v.values()) for k,v in gates.items()}
if passes['REMOVE'] and passes['PROTECT']: verdict='M5_COMBINED_READY_FOR_NOTEBOOK51_CROSSOVER'
elif passes['REMOVE']: verdict='M5_REMOVE_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER'
elif passes['PROTECT']: verdict='M5_PROTECT_ONLY_READY_FOR_NOTEBOOK51_CROSSOVER'
elif all(gates['REMOVE'][k] for k in ['finite_predictions','zero_catastrophic_regions','false_rate','wilson95_upper']): verdict='M5_LOW_COVERAGE_RESEARCH_BASELINE'
else: verdict='M5_REJECTED'
verdict_record={'verdict':verdict,'target_passes':passes,'gates':gates,'thresholds':{'REMOVE':REMOVE_PROB_THRESHOLD,'PROTECT':PROTECT_PROB_THRESHOLD},'development_only':True,'promotion_authority':False}
(OUT/'notebook50_verdict.json').write_text(json.dumps(verdict_record,indent=2))
summary_df.to_csv(TABLES/'asymmetric_target_summary.csv',index=False)
print(json.dumps(verdict_record,indent=2))


## 10. M5 construction artefacts

This notebook writes decisions and reason maps. It does not silently reinterpret M4 or modify pixels outside the frozen authority. The downstream page/mask wrapper must implement:

- start from exact M3;
- remove only final `SAFE_REMOVE` and passing `SAFE_PROTECT` connected subregions from the residual reconstruction target;
- apply frozen removal/background treatment to `SAFE_REMOVE`;
- preserve exact observed bytes at `SAFE_PROTECT`;
- retain exact M3 everywhere else.


In [ ]:
reason_codes={'FALLBACK_M3':0,'SAFE_REMOVE':1,'SAFE_PROTECT':2,'CONFLICT_FALLBACK_M3':3,'INVALID_OOD_FALLBACK_M3':4}
map_inventory=[]
for page,g in df.groupby('page'):
    h=int(g.y.max())+1; w=int(g.x.max())+1
    reason=np.zeros((h,w),np.uint8); m5=np.zeros((h,w),np.uint8)
    for name,code_value in reason_codes.items():
        q=g.m5_reason==name; reason[g.y[q].to_numpy(int),g.x[q].to_numpy(int)]=code_value
    m5[g.y[g.safe_remove].to_numpy(int),g.x[g.safe_remove].to_numpy(int)]=1
    m5[g.y[g.safe_protect].to_numpy(int),g.x[g.safe_protect].to_numpy(int)]=2
    np.savez_compressed(MAPS/f'{page}_m5_decisions.npz',reason_map=reason,m5_decision_map=m5)
    map_inventory.append({'page':page,'file':str(MAPS/f'{page}_m5_decisions.npz'),'safe_remove':int(g.safe_remove.sum()),'safe_protect':int(g.safe_protect.sum()),'fallback_m3':int((g.m5_reason=='FALLBACK_M3').sum()),'conflicts':int(g.assertion_conflict.sum())})
pd.DataFrame(map_inventory).to_csv(TABLES/'m5_map_inventory.csv',index=False)
(OUT/'reason_code_schema.json').write_text(json.dumps(reason_codes,indent=2))


## 11. Experiment identity, audit record and final upload package

The upload bundle is curated. Large working arrays remain in the run and are referenced through hashes and manifests.


In [ ]:
environment={'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'opencv':cv2.__version__,'platform':platform.platform()}
(OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.7-notebook50-asymmetric-m5','development_only':True,'promotion_authority':False,'source_record':source_record,'predictor_schema_sha256':sha256_file(INPUTS/'predictor_schema.json'),'fold_manifest_sha256':sha256_file(INPUTS/'candidate_fold_manifest.parquet'),'selected_models':selected_models,'constants':{'remove_threshold':REMOVE_PROB_THRESHOLD,'protect_threshold':PROTECT_PROB_THRESHOLD,'connectivity':CONNECTIVITY,'min_remove_run':MIN_REMOVE_RUN,'min_protect_run':MIN_PROTECT_RUN},'verdict':verdict_record,'software':environment}
(OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))

lines=['V8.7 NOTEBOOK 50 - ASYMMETRIC OBSERVABLE RESIDUAL-MASK M5 CHALLENGER','',f'Run: {OUT}',f'Verdict: {verdict}','',summary_df.to_string(index=False),'','Target passes: '+json.dumps(passes),'','Development-only. M4 remained diagnostic and was not an inference target. Exact M3 is the fallback for every non-asserted location.']
(OUT/'executive_summary.txt').write_text('
'.join(lines))

upload_files=[OUT/'executive_summary.txt',OUT/'notebook50_verdict.json',OUT/'experiment_identity.json',OUT/'source_record.json',OUT/'selected_models.json',OUT/'reason_code_schema.json',TABLES/'model_summary.csv',TABLES/'fold_model_results.csv',TABLES/'asymmetric_target_summary.csv',TABLES/'selected_region_truth_evaluation.csv',TABLES/'catastrophic_regions.csv',TABLES/'m5_map_inventory.csv',AUDIT/'calibrator_inventory.csv',INPUTS/'predictor_schema.json',INPUTS/'source_group_split_audit.csv']
for p in upload_files:
    if not p.exists(): raise RuntimeError(f'Missing required upload artefact: {p}')
    shutil.copy2(p,UPLOAD/p.name)
manifest_rows=[{'file':p.name,'bytes':p.stat().st_size,'sha256':sha256_file(p)} for p in sorted(UPLOAD.iterdir()) if p.is_file()]
pd.DataFrame(manifest_rows).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 50 curated review package. See executive_summary.txt, notebook50_verdict.json and asymmetric_target_summary.csv first. Development-only; not production authorisation.')
shutil.make_archive(str(OUT/'notebook50_upload_bundle'),'zip',UPLOAD)
print('
'.join(lines)); print('Upload bundle:',OUT/'notebook50_upload_bundle.zip')


## Outputs to inspect first

1. `executive_summary.txt`
2. `notebook50_verdict.json`
3. `tables/asymmetric_target_summary.csv`
4. `tables/selected_region_truth_evaluation.csv`
5. `tables/catastrophic_regions.csv`
6. `tables/model_summary.csv`
7. `tables/results by page/source group` additions if required by the local source schema
8. `maps/*_m5_decisions.npz`
9. `notebook50_upload_bundle.zip`

**Interpretation rule:** a REMOVE-only or PROTECT-only result is a legitimate outcome. Do not lower thresholds merely to force a combined M5.
